# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [4]:
!pip install chromadb sentence-transformers tqdm scikit-learn plotly pydantic

  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [73]:
from pathlib import Path


from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm

import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go




DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("/content/knowledge-base")
AVERAGE_CHUNK_SIZE = 500



In [64]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [65]:
class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        metadata = {
            "source": document["source"],
            "type": document["type"]
        }

        return Result(
            page_content=(
                self.headline
                + "\n\n"
                + self.summary
                + "\n\n"
                + self.original_text
            ),
            metadata=metadata
        )


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [10]:
import zipfile
with zipfile.ZipFile("/content/knowledge-base.zip", "r") as zip_ref:
    zip_ref.extractall("/content")

In [25]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [26]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [27]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [45]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: /content/knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype="auto",
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [51]:
def make_messages(document):
    return [
        {
            "role": "system",
            "content": """
You are a document chunking assistant.

Split the document into meaningful semantic chunks.

You MUST return ONLY valid JSON.

The JSON MUST have exactly this structure:

{
  "chunks": [
    {
      "headline": "short title for the chunk",
      "summary": "brief summary of the chunk",
      "original_text": "the original text belonging to this chunk"
    }
  ]
}

IMPORTANT:
- Use "headline", NOT "title".
- Use "summary" for the summary.
- Use "original_text", NOT "text".
- Do not add other field names.
- Do not use Markdown.
- Do not include ```json.
- Do not write anything before or after the JSON.
"""
        },
        {
            "role": "user",
            "content": f"""
Document:

{document['text']}
"""
        }
    ]

In [52]:
make_messages(documents[0])
# documents[:1]

[{'role': 'system',
  'content': '\nYou are a document chunking assistant.\n\nSplit the document into meaningful semantic chunks.\n\nYou MUST return ONLY valid JSON.\n\nThe JSON MUST have exactly this structure:\n\n{\n  "chunks": [\n    {\n      "headline": "short title for the chunk",\n      "summary": "brief summary of the chunk",\n      "original_text": "the original text belonging to this chunk"\n    }\n  ]\n}\n\nIMPORTANT:\n- Use "headline", NOT "title".\n- Use "summary" for the summary.\n- Use "original_text", NOT "text".\n- Do not add other field names.\n- Do not use Markdown.\n- Do not include ```json.\n- Do not write anything before or after the JSON.\n'},
 {'role': 'user',
  'content': "\nDocument:\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experi

In [61]:
import json

def process_document(document):
    messages = make_messages(document)

    # Convert messages to Qwen's chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Remove the input prompt from the output
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    reply = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("RAW RESPONSE:")
    print(reply)

    # Parse JSON
    try:
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
    except Exception as e:
        print("\nFAILED TO PARSE JSON")
        print("Response length:", len(reply))
        print("Last 500 characters:")
        print(reply[-500:])
        raise e

    return [
        chunk.as_result(document)
        for chunk in doc_as_chunks
    ]

In [66]:
process_document(documents[0])

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Company Background and Founding",
      "summary": "Insurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to disrupt the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.",
      "original_text": "Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers."
    },
    {
      "headline": "Growth and Product Expansion",
      "summary": "Insurellm experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, it had 200 employees and 12 offices across the US.",
      "original_text": "The company experienced rapid growth in its fi

[Result(page_content='Company Background and Founding\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to disrupt the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': '/content/knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Growth and Product Expansion\n\nInsurellm experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, it had 200 employees and 12 offices across the US.\n\nThe company experienced rapid growth in its first five years, expanding its product

In [77]:
#using only 10 , as it takes huge amount of time
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents[:10]):
        chunks.extend(process_document(doc))
    return chunks

In [78]:
chunks = create_chunks(documents)

 10%|█         | 1/10 [00:46<06:58, 46.51s/it]

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Company Background and Founding",
      "summary": "Insurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to disrupt the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.",
      "original_text": "Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers."
    },
    {
      "headline": "Growth and Product Expansion",
      "summary": "Insurellm experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, it had 200 employees and 12 offices across the US.",
      "original_text": "The company experienced rapid growth in its fi

 20%|██        | 2/10 [01:20<05:10, 38.86s/it]

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Why Join Insurellm?",
      "summary": "Insurellm is revolutionizing the insurance industry and has grown from a startup to a lean, profitable company with 32 employees. The company focuses on sustainable growth, operational excellence, and a remote-first culture.",
      "original_text": "Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines. After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners."
    },
    {
      "headline": "Our Culture",


 30%|███       | 3/10 [02:01<04:41, 40.16s/it]

...

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Healthllm Overview",
      "summary": "Healthllm is a comprehensive health insurance platform by Insurellm that uses AI and healthcare data analytics to streamline insurance operations.",
      "original_text": "Healthllm is Insurellm's comprehensive health insurance platform that empowers insurance providers to deliver modern, personalized health coverage. By combining advanced AI technology with healthcare data analytics, Healthllm streamlines every aspect of health insurance operations—from plan design and enrollment to claims processing and member engagement. Built for the complexities of the healthcare industry, Healthllm helps insurers reduce costs, improve member outcomes, and navigate the evolving regulatory landscape with confidence."
    },
    {
      "headline": "Key Features of Healthllm",
      "summary": "Healthllm offers a range of features including intelligent plan design, real-time eligibility verification, AI-dr

 90%|█████████ | 9/10 [07:25<00:58, 58.15s/it]

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Rellm: AI-Powered Enterprise Reinsurance Solution",
      "summary": "Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. It leverages AI to enhance risk management, decision-making, and operational efficiency.",
      "original_text": "Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility."
    },
    {
      "headline": "Key Features of Rellm",
      "summary": "Rellm offers AI-driven analy

100%|██████████| 10/10 [09:06<00:00, 54.65s/it]

RAW RESPONSE:
{
  "chunks": [
    {
      "headline": "Bizllm Overview",
      "summary": "Bizllm is an enterprise-grade commercial insurance platform designed to revolutionize how insurers serve business customers. It offers comprehensive tools for underwriting, policy administration, and risk management across multiple commercial lines.",
      "original_text": "Bizllm is Insurellm's enterprise-grade commercial insurance platform designed to revolutionize how insurers serve business customers. From small businesses to large corporations, Bizllm provides comprehensive tools for underwriting, policy administration, and risk management across multiple commercial lines including general liability, professional liability, property, workers' compensation, and cyber insurance. By leveraging AI and industry-specific data analytics, Bizllm enables commercial insurers to assess complex risks accurately, price policies competitively, and deliver exceptional service to business clients."
    },


In [79]:
print(len(chunks))

51


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [93]:
import chromadb
from sentence_transformers import SentenceTransformer

DB_NAME = "vector_db"
COLLECTION_NAME = "insurellm"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=DB_NAME)

# Delete existing collection
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME
)

texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

vectors = embedding_model.encode(
    texts,
    normalize_embeddings=True
).tolist()

collection.add(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=vectors,
)

print(f"Vectorstore created with {collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 51 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [98]:
import chromadb
import numpy as np

DB_NAME = "vector_db"
COLLECTION_NAME = "insurellm"

chroma = chromadb.PersistentClient(path=DB_NAME)

collection = chroma.get_collection(
    name=COLLECTION_NAME
)

result = collection.get(
    include=["embeddings", "documents", "metadatas"]
)

vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]

doc_types = [
    metadata["type"]
    for metadata in metadatas
]

colors = [
    ["blue", "green", "red", "orange"][
        ["products", "employees", "contracts", "company"].index(t)
    ]
    for t in doc_types
]

print("Vectors:", vectors.shape)
print("Documents:", len(documents))
print("Metadata:", len(metadatas))

Vectors: (51, 384)
Documents: 51
Metadata: 51


In [101]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:10]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [100]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [102]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [111]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.

You are given a question and a list of document chunks.
Rank ALL chunks from most relevant to least relevant to the question.

Return ONLY valid JSON in exactly this format:

{"order": [3, 1, 2]}

The numbers must be the chunk IDs provided.
Include every chunk ID exactly once.
Do not include any explanation.
"""

    user_prompt = f"""
Question:
{question}

Rank these chunks from most relevant to least relevant.

"""

    for index, chunk in enumerate(chunks):
        user_prompt += (
            f"\nCHUNK ID: {index + 1}\n"
            f"{chunk.page_content}\n"
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    reply = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("RAW RERANK RESPONSE:")
    print(reply)

    # Parse JSON
    try:
        rank_order = RankOrder.model_validate_json(reply)
    except Exception as e:
        print("Failed to parse reranker response.")
        print(reply)
        raise e

    order = rank_order.order

    # Validate IDs
    # Keep valid IDs returned by the model
    valid_order = [
        i for i in order
        if 1 <= i <= len(chunks)
    ]

    # Add any missing chunks in their original retrieval order
    missing = [
        i for i in range(1, len(chunks) + 1)
        if i not in valid_order
    ]

    final_order = valid_order + missing

    print("Model order:", order)
    print("Final order:", final_order)

    return [chunks[i - 1] for i in final_order]

In [104]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=RETRIEVAL_K
    )

    chunks = []

    for document, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        chunks.append(
            Result(
                page_content=document,
                metadata=metadata
            )
        )

    return chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [105]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [106]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Company Backgro...
Claimllm Overvi...
2025-2026 Roadm...
Vision Statemen...
Current Opportu...
Conclusion and ...
What We Offer

...
Why Join Insure...
Key Features of...
Roadmap for Cla...


In [107]:
reranked = rerank(question, chunks)

RAW RERANK RESPONSE:
{"order": [2, 8, 1, 7, 3, 4, 5, 6, 9, 10]}
Reranked order: [2, 8, 1, 7, 3, 4, 5, 6, 9, 10]


In [108]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Claimllm Overvi...
Why Join Insure...
Company Backgro...
What We Offer

...
2025-2026 Roadm...
Vision Statemen...
Current Opportu...
Conclusion and ...
Key Features of...
Roadmap for Cla...


In [109]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [112]:
reranked = rerank(question, chunks)

RAW RERANK RESPONSE:
{"order": [1, 3, 2]}
Model order: [1, 3, 2]
Final order: [1, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [113]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [114]:
reranked[0].page_content

'Company Background and Founding\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to disrupt the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.'

In [115]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [116]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [117]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [118]:
from transformers import pipeline


MODEL = "Qwen/Qwen3-1.7B"

pipe = pipeline(
    "text-generation",
    model=MODEL,
    device_map="auto",
    torch_dtype="auto",
    max_new_tokens=128,
    return_full_text=False,
    repetition_penalty=1.1,
)

pipe.model.generation_config.max_length = None





[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [121]:
def rewrite_query(question, history=None):
    if history is None:
        history = []

    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.

You are about to search a Knowledge Base to answer the user's question.

Conversation history:
{history}

Current question:
{question}

Rewrite the current question into a VERY short, specific search query.

Focus on the important entities, names, products, dates, and facts.
Do not answer the question.
Do not explain your reasoning.
Do not mention the company name unless necessary.

IMPORTANT:
Return ONLY the search query.
"""

    messages = [
        {
            "role": "system",
            "content": message,
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Move tensors to the model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Remove prompt tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    rewritten = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return rewritten

In [122]:
rewrite_query("Who won the IIOTY award?", [])

'iioty award winner'

In [124]:
def answer_question(question: str, history=None) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and retrieved context.
    """
    if history is None:
        history = []

    # 1. Rewrite question for retrieval
    query = rewrite_query(question, history)
    print("Search query:", query)

    # 2. Retrieve relevant chunks
    chunks = fetch_context(query)

    # 3. Build RAG messages
    messages = make_rag_messages(
        question,
        history,
        chunks
    )

    # 4. Convert messages to Qwen chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # 5. Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # 6. Move tensors to model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # 7. Generate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 8. Remove the original prompt from output
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, chunks

In [125]:
answer_question("Who won the IIOTY award?", [])

Search query: iioty award winner
RAW RERANK RESPONSE:
{"order": [1, 4, 2]}
Model order: [1, 4, 2]
Final order: [1, 4, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


("I don't have information about the IIOTY award or any specific winners in the provided knowledge base. The knowledge base contains details about products, company culture, and career opportunities at Insurellm, but there is no mention of an IIOTY award or its winners. If you have more context or a specific source about the IIOTY award, I'd be happy to help you explore that further.",
 [Result(page_content='2025-2026 Roadmap\n\nMarkellm plans to launch a mobile app, introduce a referral program, expand product offerings, and enhance AI matching capabilities.\n\n2025-2026 Roadmap: Q1 2025: Launch a mobile app version of Markellm, making it even easier for consumers and insurers to connect on-the-go. Introduce a referral program that rewards users for promoting Markellm to their network. Q2 2025: Expand the marketplace to include additional insurance products, such as life and health insurance. Partner with third-party data aggregators to enhance the accuracy of our AI matching capabili

In [126]:
answer_question("Who went to Manchester University?", [])

Search query: who went to manchester university
RAW RERANK RESPONSE:
{"order": [3, 1, 2]}
Model order: [3, 1, 2]
Final order: [3, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


("Based on the information provided in the knowledge base, there is no mention of any individual who attended Manchester University. The knowledge base discusses the company Insurellm, its history, products, culture, and career opportunities, but does not include any information about individuals' educational backgrounds. If you have additional context or a specific person in mind, please provide more details, and I'll do my best to assist you.",
 [Result(page_content='Company Background and Founding\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aiming to disrupt the industry. Its first product was Markellm, a marketplace connecting consumers with insurance providers.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': '/content/knowledge-base/c